# Lembar Kerja Mahasiswa (LKM)
## Tugas Penerapan — Siklus Hidup Machine Learning pada Dataset UCI

Pada praktikum sebelumnya (`lec03`) Anda menjalankan seluruh siklus hidup machine learning pada dataset **Fashion-MNIST** yang berupa **citra**. Sekarang Anda akan **menerapkan pemahaman itu pada dataset yang sepenuhnya berbeda**, yaitu dataset **tabular** dari [UCI Machine Learning Repository](https://archive.ics.uci.edu/datasets).

Tujuan tugas ini bukan mengulang langkah yang sama, melainkan menguji apakah Anda benar-benar memahami **alurnya** sehingga mampu memindahkannya ke persoalan baru dengan tantangan yang berbeda: fitur campuran numerik dan kategorikal, nilai yang hilang, dan kelas yang tidak seimbang.

---

### Identitas Mahasiswa

*Klik dua kali sel ini untuk mengedit, isi titik-titik, lalu tekan `Shift + Enter`.*

| | |
|---|---|
| **Nama** | ............................................................ |
| **NIM** | ............................................................ |
| **Nomor Presensi** | ............................................................ |
| **Kelas / Rombel** | ............................................................ |
| **Nama Dosen** | ............................................................ |
| **Tanggal Pengumpulan** | ............................................................ |

---

### Kartu Klaim Dataset

**Anda memilih sendiri datasetnya**, tetapi wajib memenuhi kriteria pada bagian P.2 dan **tidak boleh sama dengan mahasiswa lain**. Isi kartu di bawah, lalu daftarkan kepada dosen atau asisten sebelum mulai mengerjakan.

| | |
|---|---|
| **Nama dataset** | ............................................................ |
| **ID `ucimlrepo`** | ............ |
| **Tautan halaman UCI** | ............................................................ |
| **Jumlah baris × kolom** | ............ |
| **Jumlah kelas target** | ............ |
| **Alasan memilih dataset ini** | ............................................................ |
| **Tanggal klaim** | ............ |
| **Disetujui dosen/asisten** | ............ |

> **Aturan klaim.** Berlaku asas siapa cepat dia dapat. Sebelum mulai, periksa daftar klaim kelas. Bila dataset pilihan Anda sudah diambil orang lain, pilih dataset lain yang tetap memenuhi kriteria. Pekerjaan dengan dataset ganda akan dinilai hanya untuk mahasiswa yang mendaftar lebih dahulu.


---
## Capaian yang Diukur

Setelah menyelesaikan lembar kerja ini, Anda diharapkan mampu:

1. Merumuskan permasalahan pembelajaran dari dataset yang belum pernah Anda lihat.
2. Melakukan eksplorasi data dan mengenali masalah kualitas data (nilai hilang, kelas timpang, skala berbeda).
3. Membangun alur pra-pemrosesan yang menangani fitur numerik dan kategorikal sekaligus, tanpa kebocoran data.
4. Melatih, menyetel, dan membandingkan sedikitnya dua keluarga model.
5. Memilih dan menafsirkan metrik evaluasi yang sesuai dengan karakter datanya.
6. Menjelaskan perbedaan tantangan antara data citra dan data tabular.

## Petunjuk Pengerjaan

1. Jalankan sel kode **berurutan dari atas ke bawah**.
2. Sel bertanda `____` atau `# TODO` **harus Anda lengkapi sendiri**.
3. Setiap kegiatan memiliki **tabel hasil** dan **kotak jawaban** markdown. Klik dua kali untuk mengedit.
4. Semua angka pada tabel hasil harus berasal dari eksekusi di komputer Anda sendiri.
5. Karena setiap mahasiswa memakai dataset berbeda, **jawaban yang identik dengan mahasiswa lain otomatis dianggap tidak sah**.

### Perkiraan waktu

| Bagian | Perkiraan |
|---|---|
| Persiapan dan klaim dataset | 15 menit |
| Kegiatan 1–3 (masalah, eksplorasi, pembagian) | 45 menit |
| Kegiatan 4–5 (fitur dan pemodelan) | 60 menit |
| Kegiatan 6–7 (evaluasi dan refleksi) | 45 menit |

### Cara menyimpan sebagai PDF

Setelah semua sel dijalankan dan seluruh jawaban terisi:

* **JupyterLab:** `File` → `Save and Export Notebook As...` → `HTML`, lalu buka berkas HTML di browser dan cetak (`Ctrl + P`) dengan tujuan **Save as PDF**.
* **Google Colab:** `File` → `Print` → tujuan **Save as PDF**.
* Beri nama berkas: `LKM_UCI_NIM_NamaLengkap.pdf`


---
## Persiapan

### P.1 Memasang dan memuat pustaka

Dataset UCI diambil memakai paket resmi `ucimlrepo`. Bila belum terpasang, sel pertama akan memasangnya.

In [ ]:
# Jalankan sekali saja. Bila sudah terpasang, baris ini aman untuk dijalankan ulang.
%pip install -q ucimlrepo

In [ ]:
import time
import numpy as np
import pandas as pd
import plotly.express as px

from ucimlrepo import fetch_ucirepo, list_available_datasets

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, log_loss)

pd.set_option("display.max_columns", 100)
print("Semua pustaka berhasil dimuat.")

### P.2 Memilih Dataset Anda Sendiri

**Anda bebas memilih dataset apa pun dari UCI Machine Learning Repository**, sepanjang memenuhi seluruh kriteria di bawah. Tidak ada daftar yang harus diikuti — justru kemampuan memilih dataset yang tepat adalah bagian dari yang dinilai.

Jelajahi katalognya di https://archive.ics.uci.edu/datasets. Gunakan penyaring **Task: Classification** dan **Data Type: Tabular** untuk mempersempit pilihan.

#### Kriteria Wajib

Dataset Anda harus memenuhi **seluruh** butir berikut.

| No | Kriteria | Alasan |
|---|---|---|
| K1 | Berasal dari UCI ML Repository dan dapat diambil dengan `fetch_ucirepo(id=...)` | Agar kode pada lembar kerja ini berjalan tanpa perubahan |
| K2 | Merupakan tugas **klasifikasi** dengan target diskret **2 sampai 10 kelas** | Agar sebanding dengan Fashion-MNIST dan metriknya seragam |
| K3 | Berbentuk **tabular**, bukan citra, teks mentah, sinyal, atau deret waktu | Tantangan yang ingin diuji adalah data tabular |
| K4 | Jumlah baris **minimal 150** | Terlalu sedikit membuat evaluasi tidak bermakna |
| K5 | Jumlah fitur **minimal 4** | Terlalu sedikit membuat rekayasa fitur tidak ada gunanya |
| K6 | Memiliki **minimal satu** tantangan berikut: ada kolom kategorikal, **atau** ada nilai hilang, **atau** kelas timpang dengan rasio minimal 2 : 1 | Inilah yang membedakannya dari Fashion-MNIST yang bersih dan seimbang |
| K7 | **Bukan** dataset yang sudah dipakai sebagai contoh di kelas: Fashion-MNIST, MNIST, Iris, dan Wine | Agar benar-benar menjadi latihan penerapan |
| K8 | Memiliki dokumentasi variabel yang jelas pada halaman UCI | Anda perlu memahami arti tiap kolom untuk menjawab Kegiatan 1 |
| K9 | **Belum diklaim mahasiswa lain** di kelas Anda | Setiap mahasiswa wajib memakai dataset berbeda |

#### Anjuran tambahan

* Pilih domain yang **Anda pahami atau minati** — kesehatan, pertanian, keuangan, pendidikan, atau lainnya. Kegiatan 1 dan 6 menuntut Anda menalar dari sudut pandang domainnya, dan itu jauh lebih mudah bila topiknya Anda kenal.
* Dataset berukuran 500 sampai 15.000 baris adalah titik paling nyaman: cukup besar untuk bermakna, cukup kecil untuk cepat dilatih.
* Dataset dengan **campuran kolom numerik dan kategorikal** akan membuat Kegiatan 4 jauh lebih kaya dibandingkan dataset yang seluruhnya numerik.
* Hindari dataset dengan ratusan kolom pada tugas pertama; jumlah fitur yang besar membuat penafsiran hasil menjadi sulit.


#### Sel bantu — mencari dataset dan ID-nya

Gunakan sel berikut untuk menjelajah katalog `ucimlrepo` langsung dari notebook. Kolom `id` itulah yang Anda masukkan pada sel P.2.2.

In [ ]:
# Menampilkan seluruh dataset yang tersedia (daftarnya panjang)
# list_available_datasets()

# Lebih praktis: cari berdasarkan kata kunci pada nama dataset.
# Ganti kata kuncinya sesuai minat Anda, misalnya 'heart', 'plant', 'student', 'credit'.
list_available_datasets(search="heart")

#### P.2.2 Mendaftarkan pilihan Anda

Isi ketiga variabel di bawah. `SEED` diturunkan dari NIM agar hasil pembagian data Anda berbeda dari mahasiswa lain.

In [ ]:
# TODO (a): isi dengan ID dataset pilihan Anda (lihat kolom 'id' pada sel bantu di atas)
DATASET_ID = ____

# TODO (b): isi dengan tiga digit terakhir NIM Anda
NIM_3_DIGIT = ____

SEED = int(NIM_3_DIGIT)

print("ID dataset yang akan diambil :", DATASET_ID)
print("random_state pribadi (SEED)  :", SEED)

### P.3 Mengambil dataset

Sel berikut mengunduh dataset dari server UCI. Butuh koneksi internet.

In [ ]:
data_uci = fetch_ucirepo(id=DATASET_ID)

X_raw = data_uci.data.features.copy()
y_raw = data_uci.data.targets.copy()

# Bila target berupa DataFrame dengan satu kolom, ubah menjadi Series
if isinstance(y_raw, pd.DataFrame):
    nama_target = y_raw.columns[0]
    y_raw = y_raw[nama_target]
else:
    nama_target = y_raw.name

print("Nama dataset :", data_uci.metadata.name)
print("Jumlah baris :", X_raw.shape[0])
print("Jumlah fitur :", X_raw.shape[1])
print("Kolom target :", nama_target)
print("Jumlah kelas :", y_raw.nunique())
print("\nDaftar kelas :", sorted(y_raw.dropna().unique().tolist())[:15])

In [ ]:
# Deskripsi resmi dataset dari UCI
print("RINGKASAN:")
print(data_uci.metadata.abstract)
print("\n" + "=" * 70 + "\n")
print("INFORMASI TAMBAHAN:")
print(str(data_uci.metadata.additional_info.summary)[:1500])

In [ ]:
# Keterangan setiap variabel
data_uci.variables[["name", "role", "type", "description", "missing_values"]]

### P.4 Pemeriksaan Otomatis Kriteria

Sel berikut memeriksa apakah dataset pilihan Anda memenuhi kriteria K2 sampai K7. **Jalankan sebelum melanjutkan.** Bila ada butir yang GAGAL, kembali ke sel P.2.2 dan pilih dataset lain.

In [ ]:
def periksa_kriteria(X, y, nama_dataset):
    hasil = []

    n_kelas = y.nunique()
    hasil.append(("K2", "Klasifikasi dengan 2-10 kelas",
                  2 <= n_kelas <= 10, "{} kelas".format(n_kelas)))

    n_baris = len(X)
    hasil.append(("K4", "Minimal 150 baris", n_baris >= 150, "{} baris".format(n_baris)))

    n_fitur = X.shape[1]
    hasil.append(("K5", "Minimal 4 fitur", n_fitur >= 4, "{} fitur".format(n_fitur)))

    ada_kategorikal = X.select_dtypes(exclude=np.number).shape[1] > 0
    ada_hilang = X.isnull().sum().sum() > 0
    cacah = y.value_counts()
    rasio = cacah.iloc[0] / cacah.iloc[-1]
    timpang = rasio >= 2

    tantangan = []
    if ada_kategorikal:
        tantangan.append("ada kolom kategorikal")
    if ada_hilang:
        tantangan.append("ada nilai hilang")
    if timpang:
        tantangan.append("kelas timpang {:.1f}:1".format(rasio))

    hasil.append(("K6", "Minimal satu tantangan data", len(tantangan) > 0,
                  ", ".join(tantangan) if tantangan else "tidak ada tantangan"))

    terlarang = ["iris", "wine", "mnist", "fashion"]
    bersih = not any(k in nama_dataset.lower() for k in terlarang)
    hasil.append(("K7", "Bukan dataset contoh di kelas", bersih, nama_dataset))

    print("HASIL PEMERIKSAAN KRITERIA")
    print("=" * 62)
    for kode, ket, lolos, detail in hasil:
        print("{}  {:<32} {:<7} {}".format(
            kode, ket, "LULUS" if lolos else "GAGAL", detail))
    print("=" * 62)

    if all(h[2] for h in hasil):
        print("\nSemua kriteria terpenuhi. Silakan lanjut ke Kegiatan 1.")
    else:
        print("\nAda kriteria yang belum terpenuhi.")
        print("Kembali ke sel P.2.2 dan pilih dataset lain.")

    return pd.DataFrame(hasil, columns=["Kode", "Kriteria", "Lulus", "Keterangan"])


tabel_kriteria = periksa_kriteria(X_raw, y_raw, data_uci.metadata.name)

#### Tabel Verifikasi Kriteria

*Salin hasil pemeriksaan di atas ke dalam tabel berikut, dan isi K1, K8, serta K9 secara manual.*

| Kode | Kriteria | Terpenuhi? | Bukti / keterangan |
|---|---|---|---|
| K1 | Dapat diambil dengan `fetch_ucirepo` | ...... | ...... |
| K2 | Klasifikasi 2–10 kelas | ...... | ...... |
| K3 | Berbentuk tabular | ...... | ...... |
| K4 | Minimal 150 baris | ...... | ...... |
| K5 | Minimal 4 fitur | ...... | ...... |
| K6 | Minimal satu tantangan data | ...... | ...... |
| K7 | Bukan dataset contoh di kelas | ...... | ...... |
| K8 | Dokumentasi variabel jelas | ...... | ...... |
| K9 | Belum diklaim mahasiswa lain | ...... | ...... |

**Alasan Anda memilih dataset ini** (kaitkan dengan minat atau domain yang Anda pahami, minimal tiga kalimat):

> *Jawaban Anda:*
>
> ......


### P.5 Membatasi ukuran data (bila perlu)

Bila dataset Anda sangat besar, praktikum akan berjalan lambat. Sel berikut mengambil sampel acak maksimal 15.000 baris agar waktu pelatihan tetap wajar. Bila dataset Anda lebih kecil dari itu, tidak ada yang berubah.

In [ ]:
BATAS_BARIS = 15000

if len(X_raw) > BATAS_BARIS:
    idx = X_raw.sample(n=BATAS_BARIS, random_state=SEED).index
    X_raw = X_raw.loc[idx].reset_index(drop=True)
    y_raw = y_raw.loc[idx].reset_index(drop=True)
    print("Data disampel menjadi {} baris agar pelatihan lebih cepat.".format(BATAS_BARIS))
else:
    X_raw = X_raw.reset_index(drop=True)
    y_raw = y_raw.reset_index(drop=True)
    print("Data dipakai seluruhnya: {} baris.".format(len(X_raw)))

# Buang baris yang targetnya kosong
mask_valid = y_raw.notna()
X_raw, y_raw = X_raw[mask_valid].reset_index(drop=True), y_raw[mask_valid].reset_index(drop=True)

# Target diubah menjadi teks agar konsisten diperlakukan sebagai kategori
y_raw = y_raw.astype(str)

print("Ukuran akhir:", X_raw.shape, "| kelas:", y_raw.nunique())

---
# Kegiatan 1 — Merumuskan Permasalahan Pembelajaran

**Rujukan:** bagian "The Learning Problem" pada `lec03`.

Pada Fashion-MNIST, ketiga pertanyaan pembuka sudah dijawabkan untuk Anda. Kali ini **Anda sendiri** yang harus merumuskannya dari dokumentasi dataset.

### Langkah kerja

Baca keluaran sel P.3 (ringkasan, informasi tambahan, dan tabel variabel), lalu isi kotak jawaban di bawah.


### 1.1 Tabel Identifikasi Dataset

| Aspek | Isian |
|---|---|
| Nama dataset | ...... |
| Sumber / tahun | ...... |
| Domain (kesehatan, keuangan, pertanian, dll.) | ...... |
| Jumlah baris (N) | ...... |
| Jumlah fitur (D) | ...... |
| Nama kolom target | ...... |
| Jumlah kelas | ...... |


### 1.2 Pertanyaan Analisis

**A1. Target.** Apa yang ingin diprediksi pada dataset ini? Jelaskan dengan kalimat Anda sendiri, bukan menyalin abstrak UCI.

> *Jawaban Anda:*
>
> ......

**A2. Jenis tugas.** Termasuk klasifikasi biner atau multi-kelas? Sebutkan jumlah kelasnya.

> *Jawaban Anda:*
>
> ......

**A3. Tujuan (objective).** Siapa yang akan memakai model ini di dunia nyata, dan keputusan apa yang akan mereka ambil berdasarkan prediksinya?

> *Jawaban Anda:*
>
> ......

**A4. Konsekuensi kesalahan.** Untuk kasus Anda, mana yang lebih merugikan: *false positive* atau *false negative*? Berikan contoh konkret dari domainnya.

> *Jawaban Anda:*
>
> ......

**A5. Data.** Sebutkan tiga fitur yang menurut Anda paling berpengaruh terhadap target, beserta alasan logisnya (belum perlu dibuktikan, ini dugaan awal).

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 2 — Eksplorasi Data

**Rujukan:** bagian "Looking at the Data" pada `lec03`.

Pada Fashion-MNIST, eksplorasi berarti melihat gambar dan sebaran piksel. Pada data tabular, eksplorasi berarti memeriksa **tipe kolom, nilai hilang, sebaran fitur, dan keseimbangan kelas**.


### 2.1 Gambaran umum

In [ ]:
print("=== LIMA BARIS PERTAMA ===")
display(X_raw.head())

print("\n=== INFORMASI KOLOM ===")
X_raw.info()

In [ ]:
# Ringkasan statistik kolom numerik
X_raw.describe().T

### 2.2 Memisahkan kolom numerik dan kategorikal

Lengkapi bagian bertanda `____`. Inilah perbedaan pertama yang nyata dengan Fashion-MNIST: di sana **semua** fitur bertipe sama (intensitas piksel), di sini belum tentu.

In [ ]:
# TODO (a): ambil daftar nama kolom bertipe angka dan bukan angka
kolom_numerik = X_raw.select_dtypes(include=____).columns.tolist()
kolom_kategorikal = X_raw.select_dtypes(exclude=np.number).columns.tolist()

print("Jumlah kolom numerik     :", len(kolom_numerik))
print("Jumlah kolom kategorikal :", len(kolom_kategorikal))
print("\nKolom numerik     :", kolom_numerik[:15])
print("Kolom kategorikal :", kolom_kategorikal[:15])

### 2.3 Memeriksa nilai yang hilang

In [ ]:
hilang = X_raw.isnull().sum()
hilang = hilang[hilang > 0].sort_values(ascending=False)

if len(hilang) == 0:
    print("Tidak ada nilai yang hilang pada dataset ini.")
    tabel_hilang = pd.DataFrame(columns=["kolom", "jumlah_hilang", "persen"])
else:
    tabel_hilang = pd.DataFrame({
        "kolom": hilang.index,
        "jumlah_hilang": hilang.values,
        "persen": (hilang.values / len(X_raw) * 100).round(2)
    })
    display(tabel_hilang)
    display(px.bar(tabel_hilang, x="kolom", y="persen",
                   title="Persentase nilai hilang per kolom",
                   labels={"persen": "Persen hilang (%)", "kolom": "Kolom"}, height=400))

print("\nTotal sel yang hilang: {} dari {} ({:.2f}%)".format(
    int(X_raw.isnull().sum().sum()), X_raw.size,
    X_raw.isnull().sum().sum() / X_raw.size * 100))

### 2.4 Sebaran kelas target

Bagian ini penting. Pada Fashion-MNIST kelasnya seimbang sempurna (masing-masing 10%). Belum tentu demikian pada dataset Anda.

In [ ]:
sebaran = y_raw.value_counts().sort_values(ascending=False)
proporsi = (sebaran / len(y_raw) * 100).round(2)

tabel_kelas = pd.DataFrame({"jumlah": sebaran, "persen": proporsi})
display(tabel_kelas)

print("Kelas terbanyak  : {} ({:.2f}%)".format(sebaran.index[0], proporsi.iloc[0]))
print("Kelas tersedikit : {} ({:.2f}%)".format(sebaran.index[-1], proporsi.iloc[-1]))
print("Rasio timpang    : {:.1f} : 1".format(sebaran.iloc[0] / sebaran.iloc[-1]))

px.bar(tabel_kelas.reset_index(), x=tabel_kelas.index.name or "index", y="jumlah",
       title="Sebaran kelas target",
       labels={"jumlah": "Jumlah baris"}, height=420)

### 2.5 Sebaran beberapa fitur numerik

Perhatikan **rentang nilainya**. Bila satu kolom bernilai 0–1 sementara kolom lain bernilai 0–100.000, standarisasi menjadi wajib.

In [ ]:
if len(kolom_numerik) > 0:
    pilih = kolom_numerik[:4]
    for kol in pilih:
        display(px.histogram(X_raw, x=kol, color=y_raw,
                             title="Sebaran '{}' menurut kelas".format(kol),
                             barmode="overlay", opacity=0.7, height=350))

    print("\nRentang nilai tiap kolom numerik:")
    display(pd.DataFrame({
        "minimum": X_raw[kolom_numerik].min(),
        "maksimum": X_raw[kolom_numerik].max(),
        "rentang": X_raw[kolom_numerik].max() - X_raw[kolom_numerik].min()
    }).sort_values("rentang", ascending=False).head(10))
else:
    print("Dataset ini tidak memiliki kolom numerik.")

### 2.6 Tabel Hasil Pengamatan

| Aspek | Hasil |
|---|---|
| Jumlah baris setelah pembersihan | ...... |
| Jumlah kolom numerik | ...... |
| Jumlah kolom kategorikal | ...... |
| Total sel yang hilang (%) | ...... |
| Kolom dengan nilai hilang terbanyak | ...... |
| Kelas terbanyak (nama dan %) | ...... |
| Kelas tersedikit (nama dan %) | ...... |
| Rasio ketimpangan kelas | ...... : 1 |
| Kolom numerik dengan rentang terbesar | ...... |
| Kolom numerik dengan rentang terkecil | ...... |


### 2.7 Pertanyaan Analisis

**B1.** Apakah dataset Anda memiliki nilai hilang? Bila ya, sebutkan kolom mana dan berapa persen. Menurut Anda, mengapa nilai itu bisa hilang (bukan sekadar "karena tidak diisi")?

> *Jawaban Anda:*
>
> ......

**B2.** Apakah kelas target Anda seimbang? Bandingkan dengan Fashion-MNIST yang seimbang sempurna. Apa dampaknya terhadap pilihan metrik evaluasi nanti?

> *Jawaban Anda:*
>
> ......

**B3.** Lihat tabel rentang nilai pada sel 2.5. Berapa perbandingan antara rentang terbesar dan terkecil? Apa akibatnya bila fitur dipakai langsung tanpa standarisasi?

> *Jawaban Anda:*
>
> ......

**B4.** Dari histogram pada sel 2.5, apakah ada fitur yang sebarannya jelas berbeda antar-kelas? Sebutkan satu, dan jelaskan mengapa fitur itu berpotensi berguna bagi model.

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 3 — Pembagian Data

**Rujukan:** bagian "Train-Test-Validation Split" pada `lec03`.

Lengkapi bagian bertanda `____`. Perhatikan satu hal baru: karena kelas mungkin timpang, kita memakai `stratify` agar proporsi kelas terjaga pada setiap bagian.


In [ ]:
# TODO (a): pisahkan data uji 20% dengan random_state = SEED
#           gunakan stratify=y_raw agar proporsi kelas terjaga
X_tr, X_te, y_tr, y_te = train_test_split(
    X_raw, y_raw, test_size=____, random_state=____, stratify=y_raw)

# TODO (b): pisahkan data validasi 20% dari sisa data latih
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr, test_size=0.2, random_state=SEED, stratify=____)

print("Data latih   :", X_tr.shape)
print("Data validasi:", X_val.shape)
print("Data uji     :", X_te.shape)

print("\nProporsi kelas (%) pada setiap bagian:")
pd.DataFrame({
    "Latih": y_tr.value_counts(normalize=True).mul(100).round(1),
    "Validasi": y_val.value_counts(normalize=True).mul(100).round(1),
    "Uji": y_te.value_counts(normalize=True).mul(100).round(1),
})

### 3.1 Tabel Hasil Pengamatan

| Bagian | Jumlah baris | Persentase dari total |
|---|---|---|
| Latih | ...... | ...... |
| Validasi | ...... | ...... |
| Uji | ...... | ...... |

### 3.2 Pertanyaan Analisis

**C1.** Perhatikan tabel proporsi kelas pada ketiga bagian. Apakah proporsinya konsisten? Apa peran argumen `stratify` di sini?

> *Jawaban Anda:*
>
> ......

**C2.** Pada `lec03` kita tidak memakai `stratify`. Mengapa hal itu tidak menjadi masalah di sana, tetapi bisa menjadi masalah besar pada dataset Anda?

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 4 — Rekayasa Fitur dengan Pipeline

**Rujukan:** bagian "Feature Engineering" pada `lec03`.

Di Fashion-MNIST, featurisasi cukup dua langkah: ratakan gambar, lalu standarisasi. Pada data tabular kita menghadapi tiga tantangan sekaligus:

1. Nilai yang hilang harus diisi (**imputasi**).
2. Kolom numerik harus **distandarkan**.
3. Kolom kategorikal harus di-**one-hot encode**.

Ketiganya kita bungkus dalam satu `Pipeline` agar `fit` otomatis hanya terjadi pada data latih. Inilah cara paling aman mencegah **kebocoran data** (*data leakage*).


### 4.1 Menyusun pipeline pra-pemrosesan

Lengkapi bagian bertanda `____`.

In [ ]:
# Alur untuk kolom numerik: isi nilai hilang dengan median, lalu standarkan
alur_numerik = Pipeline([
    ("imputasi", SimpleImputer(strategy="median")),
    # TODO (a): tambahkan langkah standarisasi
    ("skala", ____()),
])

# Alur untuk kolom kategorikal: isi dengan nilai tersering, lalu one-hot encode
alur_kategorikal = Pipeline([
    ("imputasi", SimpleImputer(strategy="most_frequent")),
    # TODO (b): gunakan handle_unknown='ignore' agar kategori baru di data uji tidak menimbulkan error
    ("encoding", OneHotEncoder(handle_unknown=____)),
])

prapemrosesan = ColumnTransformer([
    ("num", alur_numerik, make_column_selector(dtype_include=np.number)),
    ("kat", alur_kategorikal, make_column_selector(dtype_exclude=np.number)),
])

print("Pipeline pra-pemrosesan berhasil disusun.")
prapemrosesan

### 4.2 Melihat hasil transformasi

Perhatikan berapa banyak kolom yang dihasilkan setelah one-hot encoding.

In [ ]:
# fit HANYA pada data latih
prapemrosesan.fit(X_tr)

X_tr_siap = prapemrosesan.transform(X_tr)
X_val_siap = prapemrosesan.transform(X_val)

print("Dimensi sebelum transformasi :", X_tr.shape)
print("Dimensi setelah transformasi :", X_tr_siap.shape)
print("Pertambahan kolom            :", X_tr_siap.shape[1] - X_tr.shape[1])

nama_fitur = prapemrosesan.get_feature_names_out()
print("\nContoh nama fitur hasil transformasi:")
print(list(nama_fitur[:10]))

### 4.3 Tabel Hasil Pengamatan

| Aspek | Hasil |
|---|---|
| Jumlah kolom sebelum transformasi | ...... |
| Jumlah kolom setelah transformasi | ...... |
| Pertambahan kolom akibat one-hot encoding | ...... |
| Strategi imputasi numerik yang dipakai | ...... |
| Strategi imputasi kategorikal yang dipakai | ...... |

### 4.4 Pertanyaan Analisis

**D1.** Berapa kolom bertambah setelah one-hot encoding? Dari kolom kategorikal mana pertambahan terbesar berasal? (Petunjuk: periksa `X_raw[kolom].nunique()`.)

> *Jawaban Anda:*
>
> ......

**D2.** Mengapa imputasi numerik memakai **median**, bukan rata-rata? Kapan pilihan itu penting?

> *Jawaban Anda:*
>
> ......

**D3.** Jelaskan dengan bahasa Anda sendiri: apa yang akan terjadi bila `prapemrosesan.fit()` dijalankan pada `X_raw` (seluruh data) alih-alih `X_tr` saja? Sebutkan istilah teknisnya.

> *Jawaban Anda:*
>
> ......

**D4.** Apa gunanya `handle_unknown='ignore'`? Berikan satu skenario konkret pada dataset Anda ketika argumen ini menyelamatkan program dari error.

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 5 — Pemodelan dan Optimisasi

**Rujukan:** bagian "Modeling and Optimization" pada `lec03`.

Kita akan melatih model dasar, menyetel hyperparameter regularisasi `C`, lalu membandingkannya dengan keluarga model yang berbeda.


### 5.1 Model dasar: regresi logistik

Lengkapi bagian bertanda `____`. Perhatikan bahwa pra-pemrosesan dan model digabung menjadi **satu** `Pipeline`.

In [ ]:
t0 = time.time()

model_lr = Pipeline([
    ("pra", prapemrosesan),
    # TODO (a): gunakan LogisticRegression dengan max_iter=2000 dan random_state=SEED
    ("model", LogisticRegression(max_iter=____, random_state=____)),
])

# TODO (b): latih pipeline pada data latih mentah (bukan yang sudah ditransformasi)
model_lr.fit(____, y_tr)

waktu_lr = time.time() - t0

acc_tr_lr = accuracy_score(y_tr, model_lr.predict(X_tr))
acc_val_lr = accuracy_score(y_val, model_lr.predict(X_val))

print("Lama pelatihan   : {:.1f} detik".format(waktu_lr))
print("Akurasi latih    : {:.4f}".format(acc_tr_lr))
print("Akurasi validasi : {:.4f}".format(acc_val_lr))

### 5.2 Menyetel hyperparameter regularisasi `C`

Sama seperti pada `lec03`, kita melatih beberapa model dengan nilai `C` berbeda lalu membandingkan akurasi latih dan validasinya.

> **Ingat.** `C` kecil berarti regularisasi kuat (model sederhana); `C` besar berarti regularisasi lemah (model bebas mengikuti data latih).

In [ ]:
C_vals = np.logspace(-4, 2, 12)

acc_tr_list, acc_val_list = [], []

for C in C_vals:
    m = Pipeline([
        ("pra", prapemrosesan),
        ("model", LogisticRegression(max_iter=2000, random_state=SEED, C=C)),
    ])
    m.fit(X_tr, y_tr)
    acc_tr_list.append(accuracy_score(y_tr, m.predict(X_tr)))
    acc_val_list.append(accuracy_score(y_val, m.predict(X_val)))
    print("C = {:.5f}  ->  latih {:.4f} | validasi {:.4f}".format(
        C, acc_tr_list[-1], acc_val_list[-1]))

df_C = pd.DataFrame({"C": C_vals, "Latih": acc_tr_list, "Validasi": acc_val_list}).set_index("C")

px.line(df_C, log_x=True, markers=True,
        title="Akurasi terhadap Parameter Regularisasi C",
        labels={"value": "Akurasi", "C": "Parameter Regularisasi C", "variable": "Data"},
        height=460)

In [ ]:
# TODO (c): temukan nilai C dengan akurasi VALIDASI tertinggi
idx_terbaik = int(np.argmax(____))
C_terbaik = C_vals[idx_terbaik]

print("C terbaik              : {:.5f}".format(C_terbaik))
print("Akurasi latih di C itu : {:.4f}".format(acc_tr_list[idx_terbaik]))
print("Akurasi validasi       : {:.4f}".format(acc_val_list[idx_terbaik]))
print("Selisih latih-validasi : {:.4f}".format(
    acc_tr_list[idx_terbaik] - acc_val_list[idx_terbaik]))

# Latih ulang model terbaik
model_lr = Pipeline([
    ("pra", prapemrosesan),
    ("model", LogisticRegression(max_iter=2000, random_state=SEED, C=C_terbaik)),
]).fit(X_tr, y_tr)

### 5.3 Membandingkan dengan keluarga model lain

Pada `lec03` kita membandingkan regresi logistik dengan jaringan saraf. Untuk data tabular, pembanding yang lebih lazim dan biasanya lebih kuat adalah **Random Forest** — sebuah model non-linear berbasis pohon keputusan.

In [ ]:
t0 = time.time()

model_rf = Pipeline([
    ("pra", prapemrosesan),
    ("model", RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)),
]).fit(X_tr, y_tr)

waktu_rf = time.time() - t0

acc_tr_rf = accuracy_score(y_tr, model_rf.predict(X_tr))
acc_val_rf = accuracy_score(y_val, model_rf.predict(X_val))

tabel_model = pd.DataFrame([
    {"Model": "Regresi Logistik", "Akurasi latih": acc_tr_list[idx_terbaik],
     "Akurasi validasi": acc_val_list[idx_terbaik], "Lama latih (detik)": waktu_lr},
    {"Model": "Random Forest", "Akurasi latih": acc_tr_rf,
     "Akurasi validasi": acc_val_rf, "Lama latih (detik)": waktu_rf},
])
tabel_model["Selisih latih - validasi"] = (
    tabel_model["Akurasi latih"] - tabel_model["Akurasi validasi"])

tabel_model.round(4)

### 5.4 Tabel Hasil Pengamatan

| Aspek | Nilai |
|---|---|
| Nilai `C` terbaik | ...... |
| Akurasi validasi pada `C` terbaik | ...... |
| Akurasi validasi pada `C` terkecil (0,0001) | ...... |
| Akurasi validasi pada `C` terbesar (100) | ...... |

| Model | Akurasi latih | Akurasi validasi | Lama latih (detik) | Selisih latih − validasi |
|---|---|---|---|---|
| Regresi Logistik | ...... | ...... | ...... | ...... |
| Random Forest | ...... | ...... | ...... | ...... |

### 5.5 Pertanyaan Analisis

**E1.** Perhatikan grafik pada sel 5.2. Apakah kurva validasi menunjukkan pola naik-lalu-turun seperti pada teori? Bila tidak, menurut Anda mengapa?

> *Jawaban Anda:*
>
> ......

**E2.** Di sisi mana grafik itu menunjukkan **underfitting**, dan di sisi mana **overfitting**? Sebutkan kisaran nilai `C`-nya.

> *Jawaban Anda:*
>
> ......

**E3.** Model mana yang lebih unggul pada data validasi, dan berapa selisihnya? Apakah keunggulan itu sepadan dengan tambahan waktu pelatihannya?

> *Jawaban Anda:*
>
> ......

**E4.** Bandingkan kolom "Selisih latih − validasi" kedua model. Model mana yang lebih rawan overfitting pada dataset Anda? Jelaskan mengapa hal itu masuk akal.

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 6 — Evaluasi Model

**Rujukan:** bagian "Evaluating the Model" pada `lec03`.

Kita menilai model terbaik pada **data uji** — yang sampai detik ini belum pernah disentuh. Ingat, data uji hanya boleh dipakai **sekali**.


### 6.1 Memilih model terbaik dan mengujinya

Lengkapi bagian bertanda `____`.

In [ ]:
# TODO (a): pilih model dengan akurasi VALIDASI tertinggi
if acc_val_rf > acc_val_list[idx_terbaik]:
    model_final, nama_final = model_rf, "Random Forest"
else:
    model_final, nama_final = model_lr, "Regresi Logistik"

print("Model terpilih:", nama_final)

# TODO (b): buat prediksi pada data UJI
y_pred = model_final.predict(____)

acc_uji = accuracy_score(y_te, y_pred)
f1_makro = f1_score(y_te, y_pred, average="macro")

print("\nAkurasi data uji : {:.4f}".format(acc_uji))
print("F1 makro          : {:.4f}".format(f1_makro))

### 6.2 Membandingkan dengan baseline

Angka akurasi tidak berarti apa-apa tanpa pembanding. Kita pakai dua baseline:

* **Tebakan acak** — menebak kelas secara acak merata.
* **Kelas mayoritas** — selalu menjawab kelas yang paling sering muncul.

Baseline kedua sangat penting bila kelas Anda timpang.

In [ ]:
dummy_acak = DummyClassifier(strategy="uniform", random_state=SEED).fit(X_tr, y_tr)
dummy_mayoritas = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)

acc_acak = accuracy_score(y_te, dummy_acak.predict(X_te))
acc_mayoritas = accuracy_score(y_te, dummy_mayoritas.predict(X_te))

tabel_baseline = pd.DataFrame([
    {"Pendekatan": nama_final, "Akurasi uji": acc_uji,
     "F1 makro": f1_makro},
    {"Pendekatan": "Baseline: kelas mayoritas", "Akurasi uji": acc_mayoritas,
     "F1 makro": f1_score(y_te, dummy_mayoritas.predict(X_te), average="macro", zero_division=0)},
    {"Pendekatan": "Baseline: tebakan acak", "Akurasi uji": acc_acak,
     "F1 makro": f1_score(y_te, dummy_acak.predict(X_te), average="macro", zero_division=0)},
])

display(tabel_baseline.round(4))
print("Model unggul {:.1f} poin persen atas baseline kelas mayoritas.".format(
    (acc_uji - acc_mayoritas) * 100))

### 6.3 Confusion matrix

In [ ]:
kelas_urut = sorted(y_te.unique())
cm = confusion_matrix(y_te, y_pred, labels=kelas_urut)

fig = px.imshow(cm, color_continuous_scale="Blues", text_auto=True,
                title="Confusion Matrix pada Data Uji — {}".format(nama_final),
                height=max(420, 60 * len(kelas_urut)))
fig.update_layout(
    xaxis_title="Label prediksi", yaxis_title="Label sebenarnya",
    coloraxis_showscale=False,
    xaxis=dict(tickmode="array", tickvals=np.arange(len(kelas_urut)), ticktext=kelas_urut),
    yaxis=dict(tickmode="array", tickvals=np.arange(len(kelas_urut)), ticktext=kelas_urut))
fig

In [ ]:
# Lima kekeliruan terbesar
cm_luar = cm.copy()
np.fill_diagonal(cm_luar, 0)

n_amb = min(5, int((cm_luar > 0).sum()))
if n_amb > 0:
    urut = np.dstack(np.unravel_index(np.argsort(-cm_luar, axis=None), cm.shape))[0][:n_amb]
    daftar_keliru = pd.DataFrame([
        {"Label sebenarnya": kelas_urut[i], "Diprediksi sebagai": kelas_urut[j],
         "Jumlah": int(cm[i, j])} for i, j in urut])
    display(daftar_keliru)
else:
    print("Tidak ada kekeliruan sama sekali. Periksa kembali apakah terjadi kebocoran data!")

### 6.4 Metrik per kelas

In [ ]:
# TODO (c): buat laporan klasifikasi pada data uji
laporan = classification_report(y_te, ____, output_dict=True, zero_division=0)

tabel_metrik = (pd.DataFrame(laporan).transpose()
                  .loc[kelas_urut, ["precision", "recall", "f1-score", "support"]]
                  .sort_values("f1-score"))

display(tabel_metrik.round(3))

px.bar(tabel_metrik.reset_index(), x="index", y="f1-score",
       title="F1-Score per Kelas (data uji)",
       labels={"index": "Kelas", "f1-score": "F1-Score"},
       height=440).update_xaxes(categoryorder="total ascending")

### 6.5 Tabel Hasil Pengamatan

| Aspek | Nilai |
|---|---|
| Model terpilih | ...... |
| Akurasi data uji | ...... |
| F1 makro data uji | ...... |
| Akurasi baseline kelas mayoritas | ...... |
| Akurasi baseline tebakan acak | ...... |
| Selisih model − baseline mayoritas | ...... |

| | Kelas | Precision | Recall | F1-Score |
|---|---|---|---|---|
| **F1 terendah** | ...... | ...... | ...... | ...... |
| **F1 tertinggi** | ...... | ...... | ...... | ...... |

| Peringkat | Label sebenarnya | Diprediksi sebagai | Jumlah |
|---|---|---|---|
| 1 | ...... | ...... | ...... |
| 2 | ...... | ...... | ...... |

### 6.6 Pertanyaan Analisis

**F1.** Apakah model Anda benar-benar lebih baik daripada baseline kelas mayoritas? Tunjukkan angkanya. Bila selisihnya kecil, apa artinya?

> *Jawaban Anda:*
>
> ......

**F2.** Bandingkan **akurasi** dengan **F1 makro** pada model Anda. Bila keduanya berbeda jauh, apa penyebabnya? Metrik mana yang lebih jujur untuk dataset Anda?

> *Jawaban Anda:*
>
> ......

**F3.** Kelas mana yang paling sulit bagi model (F1 terendah)? Kaitkan dengan sebaran kelas pada Kegiatan 2.4 — apakah kelas itu memang jarang muncul?

> *Jawaban Anda:*
>
> ......

**F4.** Lihat dua kekeliruan terbesar pada sel 6.3. Apakah kekeliruan itu masuk akal dari sudut pandang domainnya? (Contoh: pada data medis, dua kondisi dengan gejala mirip memang wajar tertukar.)

> *Jawaban Anda:*
>
> ......

**F5.** Kembali ke jawaban A4 pada Kegiatan 1 mengenai konsekuensi kesalahan. Berdasarkan confusion matrix, apakah model Anda melakukan jenis kesalahan yang paling merugikan itu? Apa yang akan Anda lakukan untuk menguranginya?

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 7 — Refleksi Transfer: Citra vs Tabular

Bagian ini menguji apakah Anda memahami **alurnya**, bukan sekadar menghafal langkah pada `lec03`.


### 7.1 Tabel Perbandingan

*Isi tabel berikut berdasarkan pengalaman Anda mengerjakan kedua praktikum.*

| Aspek | Fashion-MNIST (`lec03`) | Dataset UCI Anda |
|---|---|---|
| Jenis data | Citra grayscale 28×28 | ...... |
| Jumlah baris | 60.000 (dipakai 12.000) | ...... |
| Jumlah fitur setelah featurisasi | 784 | ...... |
| Tipe fitur | Seragam (intensitas piksel) | ...... |
| Nilai hilang | Tidak ada | ...... |
| Keseimbangan kelas | Seimbang sempurna | ...... |
| Langkah pra-pemrosesan | Flatten + standarisasi | ...... |
| Model terbaik | ...... | ...... |
| Akurasi uji | ...... | ...... |
| Metrik yang paling sesuai | ...... | ...... |


### 7.2 Pertanyaan Analisis

**G1.** Sebutkan **tiga langkah pra-pemrosesan** yang diperlukan pada dataset UCI Anda tetapi **tidak diperlukan** pada Fashion-MNIST. Jelaskan mengapa masing-masing menjadi perlu.

> *Jawaban Anda:*
>
> ......

**G2.** Sebutkan **satu langkah** yang diperlukan pada Fashion-MNIST tetapi tidak relevan pada dataset Anda, dan jelaskan alasannya.

> *Jawaban Anda:*
>
> ......

**G3.** Pada `lec03` jaringan saraf mengungguli regresi logistik. Pada dataset tabular Anda, apakah model non-linear (Random Forest) juga menang? Menurut Anda, mengapa untuk data tabular model berbasis pohon sering lebih unggul daripada jaringan saraf?

> *Jawaban Anda:*
>
> ......

**G4.** Bagian mana dari siklus hidup ML (L — M — O — P) yang menurut Anda **paling banyak berubah** ketika berpindah dari data citra ke data tabular? Bagian mana yang justru **tetap sama persis**?

> *Jawaban Anda:*
>
> ......

**G5.** Bila diberi dataset baru lagi minggu depan, tuliskan **daftar periksa lima langkah pertama** yang akan Anda kerjakan, berdasarkan pengalaman dua praktikum ini.

> *Jawaban Anda:*
>
> 1. ......
> 2. ......
> 3. ......
> 4. ......
> 5. ......


---
# Kesimpulan dan Refleksi

### Kesimpulan

*Tuliskan minimal lima poin kesimpulan berdasarkan hasil percobaan Anda sendiri. Sebutkan angka bila relevan, dan hindari kalimat teori umum yang bisa ditulis tanpa menjalankan kode.*

1. ......
2. ......
3. ......
4. ......
5. ......

### Refleksi

**R1.** Kesulitan terbesar apa yang Anda hadapi saat memindahkan alur `lec03` ke dataset baru ini?

> *Jawaban Anda:*
>
> ......

**R2.** Error apa yang Anda temui, dan bagaimana Anda mengatasinya? Sebutkan minimal satu.

> *Jawaban Anda:*
>
> ......

**R3.** Bila diberi waktu satu minggu lagi, apa **satu** hal pertama yang akan Anda coba untuk memperbaiki model ini? Mengapa itu yang Anda dahulukan?

> *Jawaban Anda:*
>
> ......

### Daftar Rujukan

*Tuliskan sumber yang Anda pakai: halaman UCI dataset, dokumentasi scikit-learn, artikel, atau lainnya.*

1. ......
2. ......
3. ......


---
# Rubrik Penilaian

*Bagian ini diisi oleh dosen atau asisten praktikum.*

| No | Aspek yang Dinilai | Bobot | Skor (0–100) | Nilai |
|---|---|---|---|---|
| 1 | Ketepatan pemilihan dataset, pemenuhan kriteria K1–K9, dan alasan pemilihan | 8% | | |
| 2 | Perumusan permasalahan pembelajaran (Kegiatan 1) | 15% | | |
| 3 | Kedalaman eksplorasi data (Kegiatan 2–3) | 15% | | |
| 4 | Ketepatan pipeline pra-pemrosesan (Kegiatan 4) | 15% | | |
| 5 | Pemodelan, penyetelan, dan perbandingan model (Kegiatan 5) | 15% | | |
| 6 | Evaluasi dan pemilihan metrik (Kegiatan 6) | 15% | | |
| 7 | Refleksi transfer citra vs tabular (Kegiatan 7) | 12% | | |
| 8 | Kesimpulan, refleksi, dan rujukan | 5% | | |
| | **Nilai Akhir** | **100%** | | |

**Catatan dosen:**

> ......

### Pedoman skor jawaban analisis

| Skor | Kriteria |
|---|---|
| 85–100 | Jawaban tepat, didukung angka dari percobaan sendiri, dikaitkan dengan domain datasetnya, dan menunjukkan penalaran mandiri |
| 70–84 | Jawaban tepat tetapi kurang didukung data, atau tidak dikaitkan dengan domain dataset |
| 55–69 | Jawaban sebagian benar, atau hanya mengulang teori tanpa menyentuh hasil percobaan |
| < 55 | Jawaban tidak tepat, kosong, atau menunjukkan kemiripan mencolok dengan pekerjaan mahasiswa lain |


---
# Daftar Periksa Sebelum Mengumpulkan

Centang dengan mengganti `[ ]` menjadi `[x]` (klik dua kali sel ini untuk mengedit).

- [ ] Identitas dan **kartu klaim dataset** sudah diisi lengkap
- [ ] Dataset saya sudah didaftarkan dan dikonfirmasi berbeda dari mahasiswa lain
- [ ] `DATASET_ID` dan `NIM_3_DIGIT` sudah diisi dengan data saya sendiri
- [ ] Sel P.4 menunjukkan **semua kriteria LULUS**
- [ ] Tabel verifikasi kriteria (K1–K9) sudah diisi lengkap
- [ ] Semua sel bertanda `TODO` sudah dilengkapi dan berjalan tanpa error
- [ ] Seluruh sel sudah dijalankan berurutan dari atas ke bawah
- [ ] Semua tabel hasil pengamatan sudah diisi angka dari komputer saya
- [ ] Semua pertanyaan analisis A1 sampai G5 sudah dijawab
- [ ] Tabel perbandingan citra vs tabular (7.1) sudah lengkap
- [ ] Kesimpulan minimal lima poin dan refleksi R1–R3 sudah ditulis
- [ ] Daftar rujukan sudah diisi
- [ ] Notebook sudah disimpan (`Ctrl + S`) sebelum diekspor

### Langkah ekspor PDF

1. Simpan notebook: `Ctrl + S`
2. `File` → `Save and Export Notebook As...` → `HTML`
3. Buka berkas HTML di browser, tekan `Ctrl + P`, pilih **Save as PDF**
4. Pada dialog cetak, aktifkan **Background graphics** agar grafik ikut tercetak berwarna
5. Beri nama berkas: `LKM_UCI_NIM_NamaLengkap.pdf`

> **Bila grafik Plotly tidak muncul pada hasil ekspor,** jalankan sel di bawah lalu jalankan ulang sel-sel grafik sebelum mengekspor.


In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"
print("Renderer Plotly diatur ke 'notebook'. Jalankan ulang sel grafik, lalu ekspor kembali.")

---

*Lembar Kerja Mahasiswa — Tugas Penerapan Siklus Hidup Machine Learning*
*Kelanjutan dari praktikum `lec03` (Fashion-MNIST). Dataset bersumber dari UCI Machine Learning Repository (https://archive.ics.uci.edu/datasets).*
